출발점 > 과적합을 피하는거 (상관없는 노이즈를 같이 학습시켜서)

ex) 다중공선성 억제 + 과적합 억제 역할을 하기 떄문에, vif 기준으로 전처리 안한채로 넣음

선형회귀에서만 vif 로 제거 (트리,릿지,라쏘,다항회귀 x)
vif 는 선형성 가정, 다른거는 거리,트리기반, 선형성 가정 x

상한선을 두거나 ㅌ특정값을 곱해서 계수가 너무 커지ㅣㅈ 않도록



릿지는 선형회귀의 문제를 개선 x 완화 정도...
중요하지 않은건 버리는걸 알고리즘이 알아서 함


EDA -> 지우거나 대체하는
OR 행단위 삭제 (결측치 확인)

데이터품질확인 과정에서는 LOG 변환 단계 X 

상관분석, ANOVA 는 원데이터로 하는거야

상관분석은 다중공선성확인용이라 종속변수는 안넣어도 무방

ANOVA 에서 유의미한 차이가 없으면 모형에 투입 X
(열단위의 삭제)


ANOVA 했을 떄 유의미하지 않아도 상관성이 높으면 너무 지우진마요 (교효작용 있을 수도 있음)


그 다음이 로그 변환 (모형한테 데이터주기 직전에)
그 데이터를 분리


*분리하고 70:30 그 데이터에 대해서 각각 부분에 스케일링 해야해

7 FIT TRANSFORM , 3 은 TRANSFORM


트레인 데이터에 대해서면 다중공선성 확인
만약 여기서 삭제되는 열이 생기면,,, 테스트 데이터에서도 제거해줘야함


규저,거리 기반 모델은 바로 스케일링
트리는 스케일링도 안해도 무방

파이프라인 클래스로 코드 흐름을 만들어줄 수있어
모듈


트레인데이터에 적용한 처리들을 훈련 데이터에도 자동으로 적요하는 모듈







================================
로그변환까지 딘 데이터셋


로그변환> 스케일링 > 하이퍼 파라미터>모형적합
각 모형마다 어떤 파라미턱 ㅏ있는지 고민하기


파이프라인 튜픙릉ㄹ 원소로 갖는 리스트 데이터를 준다
작업 순서대로.....
스케일링 > 학습 모형

앞에 이름 지어준거 SCALER, MODEL

작업 __ 언더바 두개 + 하이퍼파라미터 이름

릿지클래스에 ALPHA 파라미터에 하나씩테스트 하는거 -> 로그 스케일로 맞춰서...


교차검증 5회@ 설명력 위주로!

PIPELINE 이 나중에 X_TEST 데이터르 할떄 알아서 스케일링까지 해줌



ESTIMATOR 는 베스트 추정기를 받게 되는 것
GRID SC > 25개 테스트하고, 


LEARNING CURVE 에서는 원본데이터를 걸어줘야해 
ESTIMATOR 가 30 70 데이터 나눠서 알아서 스케일링까지 다 함



전달받은ㄷ ㅔ이터 10% -100%까지 8단계로 쪼개서 에러율을 점검해라


데이터들에 대해서 평균낸다 교차검증의 평균~
마지막 지점 뽑아거 최종훈련데이터


과적합은 다 도메인 기준

설명력이 세개중에 가장 작은것보다는 더 잘 나와라~ (주관적)


수치값이 아니라 그래프 모양으로 판정에서 뒤에 가서 만ㄷ날 기미가 있으면 채택하는게 나음




# [LAB 07] 지도학습 > 예측 > 선형 > 규제적용 > Ridge

## #01. 릿지 회귀 개요
- 선형회기에 벌점을 추가해서 과적합을 줄이는 방법
- 선형회귀가 너무 과하게 학습하지 않도록 계수에 브레이크를 거는 방법

### [1] 일반 선형회귀의 문제점
- 변수 수가 많음
- 변수들끼리 강하게 연관되면 다중공선성 발생
- 계수가 과도하게 커짐
- 학습 데이터에는 잘 맞지만 새로운 데이터에는 성능이 별로임
  > 릿지 회귀는 선형


### [2] 핵심 아이디어
- 계수가 너무 커지면 벌점 부과
- 모든 계수를 조금씩 줄임
- 중요 변수는 남기되, 극단적인 값 방지



### [3] 언제 쓰면 좋은가?
- 변수 수가 많을 때
- 변수 간 상관관계 높음
- 해석보다 예측 안정성 중요


### [4] 주의사항
- 다항회귀 , 릿지 ,라쏘, SDG 에서는 VIF 를 제거하지 않음
- 다항회귀의 경우 의도적으로 공선성을 생성하는 방식이기 떄문에, 다중공선성을 제거할 경우 2차항을 생성한 의미가 사라짐
- 릿지 ,라쏘, SDG는 다중공선성 해결을 위해 존재하는 방법이므로, 전처리 과정에서 VIF 필터링을 수행하게 되면 규제의 역할을 침범해서 분석 결과에 왜곡이 발생함. 즉 VIF 필터링 거치지 않고 원데이터로 돌려야함

## #02.준비작업
### [1] 패키지 참조

In [ ]:
from hossam import *
from pandas import DataFrame, read_excel
from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np
import statsmodels as sm


#데이터 표준화 모듈
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split,GridSearchCV,learning_curve

#릿지 회귀
from sklearn.linear_model import Ridge

#성능 평가 지표 모듈
from sklearn.metrics import(
  r2_score,
  mean_absolute_error,
  mean_squared_error,
  mean_absolute_percentage_error
)



### [2] 데이터 가져오기


In [ ]:
origin=load_data('fish_processed')
origin.head()

### [3] 훈련,검증 데이터 분리

In [ ]:
df = origin

yname = '무게'
x=df.drop(columns=[yname])
y=df[yname]


x_train,x_test,y_train,y_test = train_test_split(
  x,y,test_size=0.25,random_state=52
)

x_train.shape,x_test.shape,y_train.shape,y_test.shape